# Exercise: Multi-Source RAG Pipeline

## Objective
Build a complete Retrieval Augmented Generation (RAG) pipeline that collects context from multiple sources (database, documents, API), ranks them by relevance and priority, and uses them to generate AI-powered answers.

## What You'll Do
This exercise contains **11 TODOs** organized into key steps of a RAG system:

1. **Sources (TODOs 1-3)**: Retrieve context from different sources
   - TODO 1: Get data from database
   - TODO 2: Get data from documents
   - TODO 3: Get data from API

2. **Normalization (TODO 4)**: Standardize the data format across sources

3. **Collection (TODO 5)**: Combine context from all sources into one list

4. **Scoring (TODO 6)**: Calculate relevance scores based on query matches

5. **Ranking (TODO 7)**: Sort context by a combination of relevance and priority scores

6. **Selection (TODO 8)**: Keep only the top K most relevant items

7. **Formatting (TODO 9)**: Build a readable context string for the LLM

8. **LLM Integration (TODO 10)**: Call OpenAI's API with system message and context

9. **Pipeline (TODO 11)**: Connect all steps together into one complete function

## Instructions
- Run the first cell to load environment variables
- Review each TODO section and understand what data it provides or how it transforms data
- The code is partially implemented - some functions have TODO comments indicating what logic needs to be added
- The final test at the bottom will run the complete pipeline end-to-end
- Pay attention to how data flows through each step of the pipeline

## Key Concepts
- **Context**: Information from various sources that helps the LLM answer questions accurately
- **Ranking**: Prioritizing context by both relevance to the query and importance
- **RAG**: Combining retrieval (getting relevant data) with generation (creating responses)

In [1]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

True

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage


# -------------------------
# TODO 1: Source - Database
# -------------------------
def get_db_context():
    return [
        {
            "content": "Enterprise customers contribute 70% of revenue.",
            "source": "database",
            "priority": 5
        }
    ]


# -------------------------
# TODO 2: Source - Documents
# -------------------------
def get_doc_context():
    return [
        {
            "content": "Subscription pricing works best for SaaS.",
            "source": "documents",
            "priority": 3
        },
        {
            "content": "Freemium models fail in enterprise environments.",
            "source": "documents",
            "priority": 4
        }
    ]


# -------------------------
# TODO 3: Source - API
# -------------------------
def get_api_context():
    return [
        {
            "content": "Competitor recently introduced usage-based pricing.",
            "source": "api",
            "priority": 4
        }
    ]


# -------------------------
# TODO 4: Normalize Context
# -------------------------
def normalize_context(raw_context):

    normalized = []

    for item in raw_context:

        # TODO: ensure consistent schema
        normalized.append({
            "text": item["content"],
            "source": item["source"],
            "priority": item.get("priority", 1)
        })

    return normalized


# -------------------------
# TODO 5: Combine Sources
# -------------------------
def collect_context():

    raw = []
    raw.extend(get_db_context())
    raw.extend(get_doc_context())
    raw.extend(get_api_context())

    return normalize_context(raw)


# -------------------------
# TODO 6: Relevance Scoring
# -------------------------
def compute_relevance(text, query):

    score = 0

    for word in query.lower().split():
        if word in text.lower():
            score += 1

    return score


# -------------------------
# TODO 7: Ranking
# -------------------------
def rank_context(contexts, query):

    scored = []

    for c in contexts:

        relevance = compute_relevance(c["text"], query)
        priority = c["priority"]

        # TODO: combine relevance + priority
        score = relevance * 2 + priority

        scored.append((c, score))

    # TODO: sort descending
    scored.sort(key=lambda x: x[1], reverse=True)

    return [c for c, _ in scored]


# -------------------------
# TODO 8: Select Top Context
# -------------------------
def select_top_k(contexts, k=3):
    return contexts[:k]


# -------------------------
# TODO 9: Build Final Context
# -------------------------
def build_context(contexts):

    return "\n\n".join(
        [f"[{c['source'].upper()}] {c['text']}" for c in contexts]
    )


# -------------------------
# TODO 10: LLM Call
# -------------------------
def generate_answer(query, context):

    llm = ChatOpenAI(model="gpt-4o", temperature=0)

    messages = [
        SystemMessage(content="You are a business strategist."),
        HumanMessage(content=f"""
Use the context below:

{context}

Question:
{query}
""")
    ]

    return llm.invoke(messages).content


# -------------------------
# TODO 11: Full Pipeline
# -------------------------
def multi_source_rag(query):

    contexts = collect_context()

    ranked = rank_context(contexts, query)

    top_contexts = select_top_k(ranked)

    final_context = build_context(top_contexts)

    answer = generate_answer(query, final_context)

    return answer


# -------------------------
# Run Test
# -------------------------
query = "What pricing strategy should we use for enterprise customers?"

response = multi_source_rag(query)

print(response)

/Users/home/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Based on the provided context, the most suitable pricing strategy for enterprise customers would be a subscription-based model. Since enterprise customers contribute 70% of the revenue, it is crucial to adopt a strategy that aligns with their purchasing behavior and maximizes revenue potential. The documents indicate that subscription pricing works best for SaaS, which suggests that this model is effective in generating consistent revenue streams and providing value to customers over time.

Additionally, the freemium model is noted to fail in enterprise environments, likely because enterprise customers require more robust and comprehensive solutions that freemium models typically do not offer. Therefore, focusing on a subscription-based pricing strategy will likely be more successful in meeting the needs of enterprise customers and sustaining revenue growth.
